# Fine-tuning a masked language model

## Load Dataset:

In [1]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

## Load Tokenizer:

In [2]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

## Load Model:

In [3]:
from transformers import AutoModelForMaskedLM
import torch

ckpt = "distilbert-base-uncased"

model = AutoModelForMaskedLM.from_pretrained(ckpt)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## Tokenize Dataset:

In [4]:
tokenized_datasets = raw_datasets.map(
    function=lambda x: tokenizer(x['text'], truncation=True, max_length=512), 
    batched=True, 
    remove_columns=["text", "label"]
)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

## Data Collator:

In [5]:
import random
import torch
from transformers import DataCollatorForLanguageModeling


class WholeWordMaskingDataCollator:
    """Mask 15% of WHOLE words (all sub-tokens of a selected word get masked).

    Word-piece tokenization marks sub-word continuations with '##' prefix.
    We group tokens by word boundary, then randomly select 15% of words
    and mask ALL their sub-tokens.
    """

    def __init__(self, tokenizer, mlm_probability=0.15):
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability
        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id
        self.special_tokens = {
            tokenizer.cls_token_id, tokenizer.sep_token_id,
            tokenizer.pad_token_id, tokenizer.mask_token_id,
        }

    def _get_word_groups(self, input_ids):
        """Group token indices by whole word using '##' prefix detection."""
        groups = []
        current_group = []
        for idx, token_id in enumerate(input_ids):
            if token_id in self.special_tokens:
                if current_group:
                    groups.append(current_group)
                    current_group = []
                continue
            token_str = self.tokenizer.convert_ids_to_tokens(token_id)
            if token_str.startswith("##"):
                current_group.append(idx)
            else:
                if current_group:
                    groups.append(current_group)
                current_group = [idx]
        if current_group:
            groups.append(current_group)
        return groups

    def __call__(self, batch):
        # Pad sequences to the same length within the batch
        max_len = max(len(ex["input_ids"]) for ex in batch)

        padded_input_ids = []
        padded_attention_mask = []
        for ex in batch:
            pad_len = max_len - len(ex["input_ids"])
            padded_input_ids.append(ex["input_ids"] + [self.pad_token_id] * pad_len)
            padded_attention_mask.append(ex["attention_mask"] + [0] * pad_len)

        input_ids = torch.tensor(padded_input_ids, dtype=torch.long)
        attention_mask = torch.tensor(padded_attention_mask, dtype=torch.long)
        labels = input_ids.clone()

        for i in range(input_ids.shape[0]):
            word_groups = self._get_word_groups(input_ids[i].tolist())

            # Sample 15% of whole words to mask
            num_to_mask = max(1, int(len(word_groups) * self.mlm_probability))
            words_to_mask = random.sample(word_groups, min(num_to_mask, len(word_groups)))

            # Mask all sub-tokens of selected words
            for group in words_to_mask:
                for idx in group:
                    labels[i, idx] = input_ids[i, idx]       # keep original as label
                    input_ids[i, idx] = self.mask_token_id    # replace with [MASK]

            # Ignore padding in labels
            labels[i, attention_mask[i] == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


data_collator = WholeWordMaskingDataCollator(tokenizer=tokenizer, mlm_probability=0.15)

## Domain Adapt Model:

In [6]:
from transformers import TrainingArguments
from transformers import Trainer

In [7]:
args = TrainingArguments(
    output_dir="distilbert-mlm-imdb",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
    hub_model_id="tankgauravgt/distilbert-uncased-imdb-finetuned",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    bf16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["unsupervised"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [8]:
import math

In [9]:
perplexity = math.exp(trainer.evaluate()["eval_loss"])
perplexity

Training Loss,Validation Loss,Epoch
No log,0.668715,0


1.9517286693851532

In [10]:
trainer.train();

Epoch,Training Loss,Validation Loss
1,0.450493,0.406407
2,0.419500,0.398549
3,0.413014,0.395692


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
perplexity = math.exp(trainer.evaluate()["eval_loss"])
perplexity

Training Loss,Validation Loss,Epoch
0.413014,0.396416,3


1.4864875803688022